In [1]:
import pandas as pd
import numpy as np
import glob
import os
import sys
import statsmodels.stats.multitest as smm

In [9]:
def rank_and_decide(input_dir, fdr_threshold=0.05):
    """
    Ranks latent models by the strength of their strongest association.
    Domain-neutral (pure statistics).
    """
    
    files = glob.glob(os.path.join(input_dir, "*_associations.tsv"))
    
    if not files:
        print(f"Error: No association files found in {input_dir}")
        return

    print(f"--- Analysing {len(files)} models ---")
    print(f"Objective: Identify model with the strongest single signal (Max t²)")

    ranking_data = []

    for f in files:
        try:
            df = pd.read_csv(f, sep='\t')
            filename = os.path.basename(f).replace('_associations.tsv', '')
            
            if df.empty:
                continue

            # ---------------------------------------------------------
            # 1. Calculate Signal Strength (t²)
            # ---------------------------------------------------------
            # We calculate the t-statistic: Effect / SE
            # We square it (t²) to treat positive and negative correlations equally
            with np.errstate(divide='ignore', invalid='ignore'):
                df['t_stat'] = df['Effect Size'] / df['StdError']
            
            clean_t = df['t_stat'].replace([np.inf, -np.inf], np.nan).dropna()
            t_squared = clean_t ** 2

            if t_squared.empty:
                continue

            # ---------------------------------------------------------
            # 2. Extract Key Metrics
            # ---------------------------------------------------------
            # Primary: The single strongest association found in this model
            max_signal = t_squared.max() 
            
            # Secondary: Precision (Median Standard Error) - how 'noisy' is the model generally?
            median_se = df['StdError'].median()

            # Global Bias Check: 
            # In a clean model, the median t² should be ~0.45. 
            # If much higher, the model might be systematically biased (noise).
            median_signal = t_squared.median()
            bias_ratio = median_signal / 0.4549364

            # Info Only: Count significant hits
            reject, _, _, _ = smm.multipletests(
                df['P'], alpha=fdr_threshold, method='fdr_bh'
            )
            n_hits = np.sum(reject)

            ranking_data.append({
                'Model': filename,
                'Max_Signal_Strength': max_signal,
                'Median_SE': median_se,
                'Global_Bias': bias_ratio,
                'Significant_Hits': n_hits
            })
            
        except Exception as e:
            print(f"Warning: Could not process {f}. Error: {e}")

    # ---------------------------------------------------------
    # 3. Decision Logic
    # ---------------------------------------------------------
    rank_df = pd.DataFrame(ranking_data)

    if rank_df.empty:
        print("No valid data found.")
        return

    # Sort Logic:
    # 1. Highest Max Signal (The "Best" Association)
    # 2. Lowest Median SE (The most precise model)
    rank_df = rank_df.sort_values(
        by=['Max_Signal_Strength', 'Median_SE'], 
        ascending=[False, True]
    ).reset_index(drop=True)

    rank_df.index += 1
    rank_df.index.name = 'Rank'

    # ---------------------------------------------------------
    # 4. Output the Decision
    # ---------------------------------------------------------
    best_model = rank_df.iloc[0]
    
    print("\n" + "="*40)
    print("DECISION: BEST MODEL SELECTED")
    print("="*40)
    print(f"Model Name      : {best_model['Model']}")
    print(f"Peak Strength   : {best_model['Max_Signal_Strength']:.4f} (t²)")
    print(f"Precision (SE)  : {best_model['Median_SE']:.6f}")
    print(f"Significant Hits: {int(best_model['Significant_Hits'])}")
    
    # Check for systematic bias (e.g. batch effects or confounding)
    if best_model['Global_Bias'] > 1.5:
        print("\n[!] NOTE: This model has a high Global Bias score (>1.5).")
        print("    This implies many latents are slightly significant, which might indicate")
        print("    confounding factors (e.g. age/sex) rather than specific disease association.")

    # Format and Save
    output_df = rank_df.copy()
    output_df['Max_Signal_Strength'] = output_df['Max_Signal_Strength'].map('{:.4f}'.format)
    output_df['Global_Bias'] = output_df['Global_Bias'].map('{:.4f}'.format)
    output_df['Median_SE'] = output_df['Median_SE'].map('{:.6f}'.format)
    
    return output_df

In [6]:
def rank_latent_models_by_strength(output_dir, fdr_threshold=0.05):
    
    files = glob.glob(os.path.join(output_dir, "*_associations.tsv"))
    
    if not files:
        print(f"No association files found in {output_dir}")
        return

    print(f"Analysing {len(files)} models. Ranking by Peak Signal Strength...\n")
    
    ranking_data = []

    for f in files:
        df = pd.read_csv(f, sep='\t')
        filename = os.path.basename(f).replace('_associations.tsv', '')
        
        # ---------------------------------------------------------
        # 1. Recalculate FDR (Benjamini-Hochberg) - Kept for reporting
        # ---------------------------------------------------------
        if not df.empty:
            reject, pvals_corrected, _, _ = smm.multipletests(
                df['P'], 
                alpha=fdr_threshold, 
                method='fdr_bh'
            )
            n_significant_fdr = np.sum(reject)
        else:
            n_significant_fdr = 0

        # ---------------------------------------------------------
        # 2. Calculate Wald Statistic (t^2)
        # ---------------------------------------------------------
        with np.errstate(divide='ignore', invalid='ignore'):
            df['t_stat'] = df['Effect Size'] / df['StdError']
        
        # Clean infinite or NaN values
        clean_t = df['t_stat'].replace([np.inf, -np.inf], np.nan).dropna()
        t_squared = clean_t ** 2

        if not t_squared.empty:
            # The 'strength' of the model is now defined by its single best hit
            max_t2 = t_squared.max() 
            mean_t2 = t_squared.mean()
            median_t2 = t_squared.median()
        else:
            max_t2 = 0.0
            mean_t2 = 0.0
            median_t2 = 0.0

        # ---------------------------------------------------------
        # 3. Calculate Inflation (Lambda)
        # ---------------------------------------------------------
        # Lambda based on median statistic (Genomic Control)
        lambda_val = median_t2 / 0.4549364

        # ---------------------------------------------------------
        # 4. Precision (Median SE)
        # ---------------------------------------------------------
        median_se = df['StdError'].median()

        ranking_data.append({
            'Model': filename,
            'Max_Chi2_Strength': max_t2,      # NEW: The primary ranking metric
            'FDR_Significant_Hits': n_significant_fdr,
            'Mean_Chi2_Strength': mean_t2,
            'Inflation_Lambda': lambda_val,
            'Median_SE': median_se
        })

    rank_df = pd.DataFrame(ranking_data)

    # ---------------------------------------------------------
    # SORT LOGIC (UPDATED)
    # ---------------------------------------------------------
    # 1. Primary: Max Chi2 (The strongest single signal found)
    # 2. Secondary: Median SE (Precision - lower is better)
    # Note: 'FDR_Significant_Hits' is excluded from sorting to satisfy your requirement.
    
    rank_df = rank_df.sort_values(
        by=['Max_Chi2_Strength', 'Median_SE'], 
        ascending=[False, True]
    ).reset_index(drop=True)

    rank_df.index += 1
    rank_df.index.name = 'Rank'

    # Formatting for display
    output_df = rank_df.copy()
    output_df['Max_Chi2_Strength'] = output_df['Max_Chi2_Strength'].map('{:.4f}'.format)
    output_df['Mean_Chi2_Strength'] = output_df['Mean_Chi2_Strength'].map('{:.4f}'.format)
    output_df['Inflation_Lambda'] = output_df['Inflation_Lambda'].map('{:.4f}'.format)
    output_df['Median_SE'] = output_df['Median_SE'].map('{:.6f}'.format)

    return output_df

In [2]:
def rank_latent_models(output_dir, fdr_threshold=0.05):
    
    files = glob.glob(os.path.join(output_dir, "*_associations.tsv"))
    
    if not files:
        print(f"No association files found in {output_dir}")
        return

    print(f"Analysing {len(files)} models. Recalculating FDR (Benjamini-Hochberg)...\n")
    
    ranking_data = []

    for f in files:
        df = pd.read_csv(f, sep='\t')
        filename = os.path.basename(f).replace('_associations.tsv', '')
        
        # ---------------------------------------------------------
        # 1. Recalculate FDR (Benjamini-Hochberg)
        # ---------------------------------------------------------
        if not df.empty:
            reject, pvals_corrected, _, _ = smm.multipletests(
                df['P'], 
                alpha=fdr_threshold, 
                method='fdr_bh'
            )
            n_significant_fdr = np.sum(reject)
        else:
            n_significant_fdr = 0

        # ---------------------------------------------------------
        # 2. Calculate Wald Statistic (t^2)
        # ---------------------------------------------------------
        with np.errstate(divide='ignore', invalid='ignore'):
            df['t_stat'] = df['Effect Size'] / df['StdError']
        
        clean_t = df['t_stat'].replace([np.inf, -np.inf], np.nan).dropna()
        t_squared = clean_t ** 2

        mean_t2 = t_squared.mean()

        # ---------------------------------------------------------
        # 3. Calculate Inflation (Lambda)
        # ---------------------------------------------------------
        median_t2 = t_squared.median()
        lambda_val = median_t2 / 0.4549364

        # ---------------------------------------------------------
        # 4. Precision (Median SE)
        # ---------------------------------------------------------
        median_se = df['StdError'].median()

        ranking_data.append({
            'Model': filename,
            'FDR_Significant_Hits': n_significant_fdr,
            'Mean_Chi2_Strength': mean_t2,
            'Inflation_Lambda': lambda_val,
            'Median_SE': median_se
        })

    rank_df = pd.DataFrame(ranking_data)

    # Sort Logic:
    # 1. Primary: Count of FDR Hits (Descending)
    # 2. Secondary: Average Signal Strength (Descending)
    # 3. Tertiary: Precision/SE (Ascending - lower error is better)
    rank_df = rank_df.sort_values(
        by=['FDR_Significant_Hits', 'Mean_Chi2_Strength', 'Median_SE'], 
        ascending=[False, False, True]
    ).reset_index(drop=True)

    rank_df.index += 1
    rank_df.index.name = 'Rank'

    output_df = rank_df.copy()
    output_df['Mean_Chi2_Strength'] = output_df['Mean_Chi2_Strength'].map('{:.4f}'.format)
    output_df['Inflation_Lambda'] = output_df['Inflation_Lambda'].map('{:.4f}'.format)
    output_df['Median_SE'] = output_df['Median_SE'].map('{:.6f}'.format)

    return output_df


In [10]:
rank_df = rank_and_decide("/group/glastonbury/soumick/rough/dis_assoc_liv_Emma/")
rank_df

--- Analysing 12 models ---
Objective: Identify model with the strongest single signal (Max t²)

DECISION: BEST MODEL SELECTED
Model Name      : qn_latents_liver_FITPARAM_full_img_128_1_seed_filt_ancestry
Peak Strength   : 152.9553 (t²)
Precision (SE)  : 0.041781
Significant Hits: 223

[!] NOTE: This model has a high Global Bias score (>1.5).
    This implies many latents are slightly significant, which might indicate
    confounding factors (e.g. age/sex) rather than specific disease association.


,Model,Max_Signal_Strength,Median_SE,Global_Bias,Significant_Hits
Rank,,,,,
1,qn_latents_liver_FITPARAM_full_img_128_1_seed_...,152.9553,0.041781,5.6014,223
2,qn_latents_liver_FITPARAM_full_img_32_1_seed_f...,147.7603,0.030123,20.9884,104
3,qn_latents_liver_FITPARAM_full_img_64_1_seed_f...,106.5111,0.045525,6.5691,119
4,qn_latents_liver_FITPARAM_seg_128_1_seed_filt_...,86.6583,0.051607,3.6574,164
5,qn_latents_liver_FITPARAM_full_img_16_1_seed_f...,86.4683,0.044096,8.4756,29
6,qn_latents_liver_FITPARAM_full_img_8_1_seed_fi...,84.3795,0.043520,7.6568,18
7,qn_latents_liver_FITPARAM_seg_128ch_8_1_seed_f...,78.6679,0.049924,4.5162,14
8,qn_latents_liver_FITPARAM_seg_64ch_8_1_seed_fi...,64.4337,0.049464,7.1267,17
9,qn_latents_liver_FITPARAM_seg_64_1_seed_filt_a...,58.6991,0.050989,4.6199,84


In [8]:
rank_df2 = rank_latent_models("/group/glastonbury/soumick/rough/dis_assoc_liv_Emma/")
rank_df2

Analysing 12 models. Recalculating FDR (Benjamini-Hochberg)...



,Model,FDR_Significant_Hits,Mean_Chi2_Strength,Inflation_Lambda,Median_SE
Rank,,,,,
1,qn_latents_liver_FITPARAM_full_img_128_1_seed_...,223,9.9294,5.6014,0.041781
2,qn_latents_liver_FITPARAM_seg_128_1_seed_filt_...,164,6.4476,3.6574,0.051607
3,qn_latents_liver_FITPARAM_full_img_64_1_seed_f...,119,10.2899,6.5691,0.045525
4,qn_latents_liver_FITPARAM_full_img_32_1_seed_f...,104,26.8038,20.9884,0.030123
5,qn_latents_liver_FITPARAM_seg_64_1_seed_filt_a...,84,5.9820,4.6199,0.050989
6,qn_latents_liver_FITPARAM_seg_32_1_seed_filt_a...,51,6.4877,4.3632,0.051184
7,qn_latents_liver_FITPARAM_full_img_16_1_seed_f...,29,10.7186,8.4756,0.044096
8,qn_latents_liver_FITPARAM_full_img_8_1_seed_fi...,18,12.3880,7.6568,0.043520
9,qn_latents_liver_FITPARAM_seg_64ch_8_1_seed_fi...,17,10.1474,7.1267,0.049464


In [4]:
rank_df

,Model,FDR_Significant_Hits,Mean_Chi2_Strength,Inflation_Lambda,Median_SE
Rank,,,,,
1,qn_latents_liver_FITPARAM_full_img_128_1_seed_...,223,9.9294,5.6014,0.041781
2,qn_latents_liver_FITPARAM_seg_128_1_seed_filt_...,164,6.4476,3.6574,0.051607
3,qn_latents_liver_FITPARAM_full_img_64_1_seed_f...,119,10.2899,6.5691,0.045525
4,qn_latents_liver_FITPARAM_full_img_32_1_seed_f...,104,26.8038,20.9884,0.030123
5,qn_latents_liver_FITPARAM_seg_64_1_seed_filt_a...,84,5.9820,4.6199,0.050989
6,qn_latents_liver_FITPARAM_seg_32_1_seed_filt_a...,51,6.4877,4.3632,0.051184
7,qn_latents_liver_FITPARAM_full_img_16_1_seed_f...,29,10.7186,8.4756,0.044096
8,qn_latents_liver_FITPARAM_full_img_8_1_seed_fi...,18,12.3880,7.6568,0.043520
9,qn_latents_liver_FITPARAM_seg_64ch_8_1_seed_fi...,17,10.1474,7.1267,0.049464
